In [34]:
import cudf
import re
import string
import contractions
from textblob import TextBlob
import emoji
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/grv06sid/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [43]:
df = cudf.read_csv('datasets/Quora Questions Pair Dataset/train.csv')

In [44]:
df.sample(5)

,id,qid1,qid2,question1,question2,is_duplicate
258755,258755,374390,83596,What is this boys haircut called?,What is this haircut called?,0
287170,287170,54800,147040,What will be the effect of India scrapping Rs....,What will be effect on stock exchange of India...,1
100976,100976,167362,167363,What are the basic and required things hr may ...,Why can I find my fb friend Reshmi addy?,0
181745,181745,278349,12804,Which is the richest man in the world?,Who is the richest man?,1
176953,176953,229812,272139,Why aren't we hearing about Donald Trump's all...,Is it true that Donald Trump raped a 13 year o...,1


In [11]:
def remove_html(text):
    pattern = re.compile(r'<.*?>')
    return pattern.sub('',text)

In [12]:
def remove_url(text):
    pattern = re.compile(r'http\S+|www\.\S+')
    return pattern.sub('',text)

In [14]:
def remove_punctuations(text):
    return text.translate(str.maketrans('','',string.punctuation))

In [15]:
def remove_multiple_space(text):
    pattern = re.compile(r'\s+')
    return pattern.sub(' ', text)

In [24]:
def correct_chat_words(text):
    chat_abbreviations = {
        "A3": "Anytime, Anywhere, Anyplace",
        "ADIH": "Another Day In Hell",
        "AFK": "Away From Keyboard",
        "AFAIK": "As Far As I Know",
        "ASAP": "As Soon As Possible",
        "ASL": "Age, Sex, Location",
        "ATK": "At The Keyboard",
        "ATM": "At The Moment",
        "BAE": "Before Anyone Else",
        "BAK": "Back At Keyboard",
        "BBL": "Be Back Later",
        "BBS": "Be Back Soon",
        "BFN": "Bye For Now",
        "B4N": "Bye For Now",
        "BRB": "Be Right Back",
        "BRUH": "Bro",
        "BRT": "Be Right There",
        "BSAAW": "Big Smile And A Wink",
        "BTW": "By The Way",
        "BWL": "Bursting With Laughter",
        "CSL": "Can’t Stop Laughing",
        "CU": "See You",
        "CUL8R": "See You Later",
        "CYA": "See You",
        "DM": "Direct Message",
        "FAQ": "Frequently Asked Questions",
        "FC": "Fingers Crossed",
        "FIMH": "Forever In My Heart",
        "FOMO": "Fear Of Missing Out",
        "FR": "For Real",
        "FWIW": "For What It's Worth",
        "FYP": "For You Page",
        "FYI": "For Your Information",
        "G9": "Genius",
        "GAL": "Get A Life",
        "GG": "Good Game",
        "GMTA": "Great Minds Think Alike",
        "GN": "Good Night",
        "GOAT": "Greatest Of All Time",
        "GR8": "Great!",
        "HBD": "Happy Birthday",
        "IC": "I See",
        "ICQ": "I Seek You",
        "IDC": "I Don’t Care",
        "IDK": "I Don't Know",
        "IFYP": "I Feel Your Pain",
        "ILU": "I Love You",
        "ILY": "I Love You",
        "IMHO": "In My Honest Opinion",
        "IMU": "I Miss You",
        "IMO": "In My Opinion",
        "IOW": "In Other Words",
        "IRL": "In Real Life",
        "IYKYK": "If You Know, You Know",
        "JK": "Just Kidding",
        "KISS": "Keep It Simple, Stupid",
        "L": "Loss",
        "L8R": "Later",
        "LDR": "Long Distance Relationship",
        "LMK": "Let Me Know",
        "LMAO": "Laughing My Ass Off",
        "LOL": "Laughing Out Loud",
        "LTNS": "Long Time No See",
        "M8": "Mate",
        "MFW": "My Face When",
        "MID": "Mediocre",
        "MRW": "My Reaction When",
        "MTE": "My Thoughts Exactly",
        "NVM": "Never Mind",
        "NRN": "No Reply Necessary",
        "NPC": "Non-Player Character",
        "OIC": "Oh I See",
        "OP": "Overpowered",
        "PITA": "Pain In The Ass",
        "POV": "Point Of View",
        "PRT": "Party",
        "PRW": "Parents Are Watching",
        "ROFL": "Rolling On The Floor Laughing",
        "ROFLOL": "Rolling On The Floor Laughing Out Loud",
        "ROTFLMAO": "Rolling On The Floor Laughing My Ass Off",
        "RN": "Right Now",
        "SK8": "Skate",
        "STATS": "Your Sex And Age",
        "SUS": "Suspicious",
        "TBH": "To Be Honest",
        "TFW": "That Feeling When",
        "THX": "Thank You",
        "TIME": "Tears In My Eyes",
        "TLDR": "Too Long, Didn’t Read",
        "TNTL": "Trying Not To Laugh",
        "TTFN": "Ta-Ta For Now!",
        "TTYL": "Talk To You Later",
        "U": "You",
        "U2": "You Too",
        "U4E": "Yours For Ever",
        "W": "Win",
        "W8": "Wait",
        "WB": "Welcome Back",
        "WTF": "What The Fuck",
        "WTG": "Way To Go!",
        "WUF": "Where Are You From?",
        "WYD": "What You Doing?",
        "WYWH": "Wish You Were Here",
        "ZZZ": "Sleeping, Bored, Tired"
    }
    new_text = []
    for word in text.split():
        word = word.replace(',','')
        word = word.replace('.','')
        word = word.upper()
        if word in chat_abbreviations.keys():
            new_text.append(chat_abbreviations[word])
        else:
            new_text.append(word)
    return " ".join(new_text).lower()

In [19]:
def expand_contractions(text):
    return contractions.fix(text)

In [27]:
def correct_spelling(text):
    txt_blob = TextBlob(text)
    return txt_blob.correct()

In [31]:
def remove_stopwords(text):
    stopwords = stopwords.words('english')
    filtered_words = [word for word in text.split() if word not in stopwords]
    return " ".join(filtered_words)

In [35]:
def handle_emojis(text):
    return emoji.demojize(text)

In [36]:
def remove_digits(text):
    pattern = re.compile(r'\d+')
    return pattern.sub('',text)

In [37]:
def lower_case(text):
    return text.lower()

In [38]:
def preprocess_text(text):
    text = remove_html(text)
    text = remove_url(text)
    text = remove_punctuations(text)
    text = remove_multiple_space(text)
    text = correct_chat_words(text)
    text = expand_contractions(text)
    text = correct_spelling(text)
    text = remove_stopwords(text)
    text = handle_emojis(text)
    text = remove_digits(text)
    text = lower_case(text)
    return text

In [45]:
df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 404290 entries, 0 to 404289
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   id            404290 non-null  object
 1   qid1          404290 non-null  object
 2   qid2          404290 non-null  object
 3   question1     404290 non-null  object
 4   question2     404288 non-null  object
 5   is_duplicate  404290 non-null  object
dtypes: object(6)
memory usage: 62.4+ MB


In [46]:
df['question1'] = df['question1'].apply(preprocess_text)

ValueError: user defined function compilation failed.